# Решения: Практика: корректность и бенчмарк сортировок

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import math
import statistics
import time
import pandas as pd


def find_csv(name):
    for path in (Path(name), Path("../../data") / name, Path("../data") / name):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f"{name} не найден рядом с ноутбуком или в data/")


unsorted_df = pd.read_csv(find_csv("bank_transactions_unsorted.csv"))
by_id_df = pd.read_csv(find_csv("bank_transactions_sorted_by_txn_id.csv"))
by_amount_df = pd.read_csv(find_csv("bank_transactions_sorted_by_amount.csv"))
tiny_df = pd.read_csv(find_csv("bank_transactions_tiny.csv"))
COLS = ["txn_id", "amount", "day", "risk_score"]
unsorted_txns = list(unsorted_df[COLS].itertuples(index=False, name=None))
id_txns = list(by_id_df[COLS].itertuples(index=False, name=None))
amount_txns = list(by_amount_df[COLS].itertuples(index=False, name=None))
tiny_txns = list(tiny_df[COLS].itertuples(index=False, name=None))
id_list = [row[0] for row in id_txns]
amount_list = [row[1] for row in amount_txns]
assert id_list == sorted(id_list)
assert amount_list == sorted(amount_list)
print(f"Загружено {len(unsorted_txns)} транзакций; поля кортежа: {COLS}")


## Урок. 1–2. Gate и таймер

In [ ]:
def linear_search(values, target):
    for index, value in enumerate(values):
        if value == target:
            return index
    return -1


def binary_search(values, target):
    left, right = 0, len(values) - 1
    while left <= right:
        mid = (left + right) // 2
        if values[mid] == target:
            return mid
        if values[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1


def lower_bound(values, target):
    left, right = 0, len(values)
    while left < right:
        mid = (left + right) // 2
        if values[mid] < target:
            left = mid + 1
        else:
            right = mid
    return left


def upper_bound(values, target):
    left, right = 0, len(values)
    while left < right:
        mid = (left + right) // 2
        if values[mid] <= target:
            left = mid + 1
        else:
            right = mid
    return left


def selection_sort(values):
    result = list(values)
    for i in range(len(result)):
        smallest = i
        for j in range(i + 1, len(result)):
            if result[j] < result[smallest]:
                smallest = j
        result[i], result[smallest] = result[smallest], result[i]
    return result


def merge_sorted(left, right):
    i = j = 0
    result = []
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    return result + list(left[i:]) + list(right[j:])


def merge_sort(values):
    if len(values) <= 1:
        return list(values)
    mid = len(values) // 2
    return merge_sorted(merge_sort(values[:mid]), merge_sort(values[mid:]))


def median_runtime(function, values, repeats=3):
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        function(list(values))
        samples.append(time.perf_counter() - start)
    return statistics.median(samples)

cases = [[], [1], [2, 1], [3, 1, 3], list(range(30, -1, -1))]
gate = [fn(case) == sorted(case) for case in cases for fn in (selection_sort, merge_sort)]
probe_time = median_runtime(sorted, amount_list[:100])
assert all(gate) and probe_time >= 0


## Урок. 3–4. Таблица

In [ ]:
sizes = [80, 160, 320, 640]
rows = []
for n in sizes:
    values = [r[1] for r in unsorted_txns[:n]]
    rows.append({"n": n, "selection_s": median_runtime(selection_sort, values), "merge_s": median_runtime(merge_sort, values)})
benchmark = pd.DataFrame(rows)
benchmark["selection_per_n2"] = benchmark["selection_s"] / benchmark["n"].pow(2)
benchmark["merge_per_nlogn"] = benchmark["merge_s"] / (benchmark["n"] * benchmark["n"].map(math.log2))
assert len(benchmark) == 4


## Урок. 5–7. Baseline и повторы

In [ ]:
builtin_s = median_runtime(sorted, amount_list[:640], 5)
def safe_ratio(a, b): return a / b if b else 0.0
growth_selection = safe_ratio(benchmark.iloc[-1].selection_s, benchmark.iloc[0].selection_s)
growth_merge = safe_ratio(benchmark.iloc[-1].merge_s, benchmark.iloc[0].merge_s)
repeat_table = [median_runtime(sorted, amount_list[:640], 1) for _ in range(5)]
assert len(repeat_table) == 5


## Урок. 8–9. Вывод

In [ ]:
BENCH_NOTE = "Таблица показывает тренд: ручной selection растёт ближе к O(n²), а merge — к O(n log n). Отдельный замер шумит из-за машины и планировщика, поэтому использована median повторов. Эксперимент согласуется с моделью, но не доказывает Big O. Для production выбираем встроенный sorted как протестированный baseline."
acceptance = {"correct": all(gate), "four_sizes": len(benchmark) == 4, "median": True, "baseline": builtin_s >= 0, "honest_note": len(BENCH_NOTE) >= 240}
assert set(acceptance.values()) == {True}


## ДЗ. Part A

In [ ]:
risk_rows = []
for n in (100, 300, 600):
    values = [r[3] for r in unsorted_txns[:n]]
    risk_rows.append([n, median_runtime(selection_sort, values), median_runtime(merge_sort, values)])
ordered = amount_list[:500]; reversed_values = list(reversed(ordered))
shape_times = {"ordered": median_runtime(merge_sort, ordered), "reversed": median_runtime(merge_sort, reversed_values)}
built_rows = [(n, median_runtime(sorted, amount_list[:n])) for n in (100, 300, 600)]
assert len(risk_rows) == len(built_rows) == 3


## ДЗ. Challenge

In [ ]:
def selection_sort_count(values):
    result = list(values); comparisons = 0
    for i in range(len(result)):
        smallest = i
        for j in range(i + 1, len(result)):
            comparisons += 1
            if result[j] < result[smallest]: smallest = j
        result[i], result[smallest] = result[smallest], result[i]
    return result, comparisons

METHOD_NOTE = "Перед серией нужен короткий warm-up. Все алгоритмы получают одинаковый input, созданный до таймера; каждый запуск работает с копией. Используются repeats и median, а не минимум одного запуска. Ограничения: фоновые процессы, версия Python и малый диапазон n влияют на числа, поэтому интерпретируем форму роста, а не абсолютный рекорд."
assert selection_sort_count([4, 3, 2, 1])[1] == 6 and len(METHOD_NOTE) >= 240
